In [5]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter, defaultdict
from gensim.parsing.preprocessing import STOPWORDS
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel
from gensim.models.phrases import Phraser
import pyLDAvis
import pyLDAvis.gensim_models
import os
from bertopic import BERTopic
import math
import concurrent.futures

csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
base_models_dir = '../data/models'
base_vis_dir = '../results/topic_modeling/visualizations'

yelp_stopwords = [
    'food', 'good', 'place', 'service', 'restaurant', 'great', 'time', 
    'go', 'back', 'really', 'just', 'like', 'get', 'one', 'would', 
    'ive', 'even', 'also', 'always', 'got', 'came', 'went', 'us', 'im', 'much'
]
stopwords_list = list(STOPWORDS) + [''] + yelp_stopwords

CATEGORIES = {
    'fast_food':   'Fast Food',
    'steakhouses': 'Steakhouses',
    'mexican': 'Mexican',
    'pubs': 'Pubs',
    'burguers': 'Burguers'
}

In [3]:
def evaluate_coherence_and_diversity(input_csv, models_dir, categories_dict, stopwords_list, sample_size=100000):
    """
    Evalúa la coherencia (C_v) y la diversidad de los tópicos generados por modelos LDA y BERTopic previamente entrenados para diferentes categorías.

    Args:
        input_csv (str): Ruta al archivo CSV que contiene las reviews para calcular la coherencia.
        models_dir (str): Directorio base donde se encuentran guardados los modelos entrenados.
        categories_dict (dict): Diccionario con las categorías a evaluar. Las claves deben coincidir con los nombres de las carpetas en models_dir.
        stopwords_list (list<str>): Lista de palabras a excluir durante la tokenización de los textos de referencia.
        sample_size (int): Número de muestras aleatorias a extraer del CSV para crear el corpus de referencia.
    """
    
    def calculate_topic_diversity(topic_words_list):
        """
        Calcula la proporción de palabras únicas respecto al total de palabras en una lista de tópicos.

        Args:
            topic_words_list (list<list<str>>): Lista donde cada elemento es una lista de palabras que representan un tópico.
        """
        if not topic_words_list:
            return 0.0
        all_words = [word for topic in topic_words_list for word in topic]
        unique_words = set(all_words)
        return len(unique_words) / len(all_words)

    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=sample_size, seed=42)
        .with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(' ')
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
    )
    reference_texts = sample_df['tokens'].to_list()
    del sample_df
    
    global_dict = Dictionary(reference_texts)
    top_n_words = 10
    results = []

    for category_key in categories_dict.keys():
        print(f'\nEvaluando categoría: {category_key} ---')
        
        lda_path = os.path.join(models_dir, category_key, 'lda', 'lda_model.gensim')
        dict_path = os.path.join(models_dir, category_key, 'lda', 'dictionary.dict')
        bigram_path = os.path.join(models_dir, category_key, 'lda', 'bigram_model.pkl')
        bertopic_path = os.path.join(models_dir, category_key, 'bertopic', 'bertopic_model.pkl') 

        # Evaluación LDA
        if os.path.exists(lda_path) and os.path.exists(dict_path) and os.path.exists(bigram_path):
            try:
                lda = LdaMulticore.load(lda_path)
                dictionary = Dictionary.load(dict_path)
                bigram_model = Phraser.load(bigram_path)
                
                lda_reference_texts = [bigram_model[doc] for doc in reference_texts]
                
                lda_topics = []
                for topic_id in range(lda.num_topics):
                    words = [dictionary[word_id] for word_id, _ in lda.get_topic_terms(topic_id, topn=top_n_words)]
                    lda_topics.append(words)

                lda_diversity = calculate_topic_diversity(lda_topics)
                
                cm_lda = CoherenceModel(
                    topics=lda_topics, 
                    texts=lda_reference_texts,
                    dictionary=dictionary, 
                    coherence='c_v'
                )
                lda_coherence = cm_lda.get_coherence()
                
                results.append({
                    'Categoría': category_key,
                    'Modelo': 'LDA',
                    'Coherencia (C_v)': round(lda_coherence, 4),
                    'Diversidad': round(lda_diversity, 4)
                })
                print(f'LDA evaluado exitosamente.')
            except Exception as e:
                print(f'Error evaluando LDA para {category_key}: {e}')
        else:
            print(f'Faltan archivos de LDA para {category_key}.')

        # Evaluación BERTopic
        if os.path.exists(bertopic_path):
            try:
                bertopic_model = BERTopic.load(bertopic_path)
                
                bertopic_topics = []
                for topic_id in bertopic_model.get_topic_info()['Topic']:
                    if topic_id != -1: 
                        rep = bertopic_model.get_topic(topic_id)
                        if rep:
                            words = [word for word, _ in rep[:top_n_words]]
                            bertopic_topics.append(words)

                bertopic_diversity = calculate_topic_diversity(bertopic_topics)

                valid_bertopic_topics = []
                for topic in bertopic_topics:
                    valid_words = []
                    for phrase in topic:
                        sub_words = str(phrase).split(' ') 
                        
                        for w in sub_words:
                            if w in global_dict.token2id and w not in valid_words:
                                valid_words.append(w)
                                
                    if len(valid_words) >= 2:
                        valid_bertopic_topics.append(valid_words)

                bertopic_coherence = 0.0
                if valid_bertopic_topics:
                    try:
                        cm_bertopic = CoherenceModel(
                            topics=valid_bertopic_topics, 
                            texts=reference_texts, 
                            dictionary=global_dict, 
                            coherence='c_v'
                        )
                        bertopic_coherence = cm_bertopic.get_coherence()
                    except Exception as e:
                        print(f'No se pudo calcular la coherencia de BERTopic: {e}')

                results.append({
                    'Categoría': category_key,
                    'Modelo': 'BERTopic',
                    'Coherencia (C_v)': round(bertopic_coherence, 4), 
                    'Diversidad': round(bertopic_diversity, 4)
                })
                print(f'BERTopic evaluado exitosamente.')
            except Exception as e:
                print(f'Error evaluando BERTopic para {category_key}: {e}')
        else:
            print(f'Faltan archivos de BERTopic para {category_key}.')

    df_results = pl.DataFrame(results)
    return df_results

final_metrics_df = evaluate_coherence_and_diversity(input_csv=csv_reviews, models_dir=base_models_dir, categories_dict=CATEGORIES, stopwords_list=stopwords_list)

print("\nResultados Finales:")
print(final_metrics_df)


Evaluando categoría: fast_food ---


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_for

  LDA evaluado exitosamente.


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_for

BERTopic evaluado exitosamente.

Evaluando categoría: steakhouses ---


...

BERTopic evaluado exitosamente.

Evaluando categoría: pubs ---


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_for

  LDA evaluado exitosamente.


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=52648) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_for

BERTopic evaluado exitosamente.

Evaluando categoría: burguers ---
  [Aviso] Faltan archivos de LDA para burguers.
Faltan archivos de BERTopic para burguers.

Resultados Finales:
shape: (8, 4)
┌─────────────┬──────────┬──────────────────┬────────────┐
│ Categoría   ┆ Modelo   ┆ Coherencia (C_v) ┆ Diversidad │
│ ---         ┆ ---      ┆ ---              ┆ ---        │
│ str         ┆ str      ┆ f64              ┆ f64        │
╞═════════════╪══════════╪══════════════════╪════════════╡
│ fast_food   ┆ LDA      ┆ 0.5295           ┆ 0.56       │
│ fast_food   ┆ BERTopic ┆ 0.4163           ┆ 0.757      │
│ steakhouses ┆ LDA      ┆ 0.559            ┆ 0.53       │
│ steakhouses ┆ BERTopic ┆ 0.4043           ┆ 0.7723     │
│ mexican     ┆ LDA      ┆ 0.572            ┆ 0.55       │
│ mexican     ┆ BERTopic ┆ 0.3906           ┆ 0.7457     │
│ pubs        ┆ LDA      ┆ 0.5536           ┆ 0.51       │
│ pubs        ┆ BERTopic ┆ 0.3866           ┆ 0.7496     │
└─────────────┴──────────┴──────────────

In [6]:
def generate_visualizations(input_csv, models_dir, vis_dir, categories_dict, stopwords_list, sample_size=50000):
    """
    Genera y guarda en disco visualizaciones interactivas (HTML) para los modelos 
    de tópicos generados por LDA (pyLDAvis) y BERTopic (Plotly).

    Args:
        input_csv (str): Ruta al archivo CSV con los textos originales para el corpus.
        models_dir (str): Directorio base donde están guardados los modelos entrenados.
        vis_dir (str): Directorio base donde se guardarán los archivos HTML resultantes.
        categories_dict (dict): Diccionario de categorías a procesar.
        stopwords_list (list<str>): Lista de palabras vacías para omitir en la tokenización.
        sample_size (int): Tamaño de la muestra a extraer del CSV.
    """
    
    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=sample_size, seed=42)
        .with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(' ')
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
    )
    
    tokenized_texts = sample_df['tokens'].to_list()
    del sample_df  

    for category_key in categories_dict.keys():
        print(f'\nVisualizaciones para: {category_key} ---')
        
        cat_model_dir = os.path.join(models_dir, category_key)
        lda_path = os.path.join(cat_model_dir, 'lda', 'lda_model.gensim')
        dict_path = os.path.join(cat_model_dir, 'lda', 'dictionary.dict')
        bigram_path = os.path.join(cat_model_dir, 'lda', 'bigram_model.pkl')
        bertopic_path = os.path.join(cat_model_dir, 'bertopic', 'bertopic_model.pkl') 
        
        cat_vis_dir = os.path.join(vis_dir, category_key)
        os.makedirs(cat_vis_dir, exist_ok=True)
        
        lda_html_output = os.path.join(cat_vis_dir, 'lda_intertopic_map.html')
        intertopic_html_output = os.path.join(cat_vis_dir, 'bertopic_intertopic_map.html')
        words_html_output = os.path.join(cat_vis_dir, 'bertopic_topic_words.html')

        # Visualización LDA
        if os.path.exists(lda_path) and os.path.exists(dict_path) and os.path.exists(bigram_path):
            try:
                lda = LdaMulticore.load(lda_path)
                dictionary = Dictionary.load(dict_path)
                bigram_model = Phraser.load(bigram_path)
                
                cat_tokenized_texts = [bigram_model[doc] for doc in tokenized_texts]
                bow_corpus = [dictionary.doc2bow(text) for text in cat_tokenized_texts]
                
                vis_data_lda = pyLDAvis.gensim_models.prepare(
                    lda, 
                    bow_corpus, 
                    dictionary, 
                    mds='pcoa',
                    R=10
                )
                pyLDAvis.save_html(vis_data_lda, lda_html_output)
                print(f'LDA visualización guardada en: {lda_html_output}')
            except Exception as e:
                print(f'Generando LDA vis para {category_key}: {e}')
        else:
            print(f'Faltan archivos de LDA para {category_key}, saltando...')

        # Visualización BERTOpic
        if os.path.exists(bertopic_path):
            try:
                bertopic_model = BERTopic.load(bertopic_path)
                
                fig_intertopic = bertopic_model.visualize_topics(top_n_topics=15)
                fig_words = bertopic_model.visualize_barchart(top_n_topics=15, n_words=5)
                    
                fig_intertopic.write_html(intertopic_html_output)
                fig_words.write_html(words_html_output)
                
                print(f'BERTopic visualizaciones guardadas en: {cat_vis_dir}')
            except Exception as e:
                print(f'Generando BERTopic vis para {category_key}: {e}')
        else:
            print(f'Faltan archivos de BERTopic para {category_key}')

generate_visualizations(input_csv=csv_reviews, models_dir=base_models_dir, vis_dir=base_vis_dir, categories_dict=CATEGORIES,stopwords_list=stopwords_list)


Visualizaciones para: fast_food ---
  [Éxito] LDA visualización guardada en: ../results/topic_modeling/visualizations/fast_food/lda_intertopic_map.html
BERTopic visualizaciones guardadas en: ../results/topic_modeling/visualizations/fast_food

Visualizaciones para: steakhouses ---
  [Éxito] LDA visualización guardada en: ../results/topic_modeling/visualizations/steakhouses/lda_intertopic_map.html
BERTopic visualizaciones guardadas en: ../results/topic_modeling/visualizations/steakhouses

Visualizaciones para: mexican ---
  [Éxito] LDA visualización guardada en: ../results/topic_modeling/visualizations/mexican/lda_intertopic_map.html
BERTopic visualizaciones guardadas en: ../results/topic_modeling/visualizations/mexican

Visualizaciones para: pubs ---
  [Éxito] LDA visualización guardada en: ../results/topic_modeling/visualizations/pubs/lda_intertopic_map.html
BERTopic visualizaciones guardadas en: ../results/topic_modeling/visualizations/pubs

Visualizaciones para: burguers ---
  [Avis